# Business Entity Resolution — Clean Final Pipeline

Run the notebook from top to bottom once. It trains the model, generates the two required TSV files, and performs basic output checks.

**Required input:** `student_resource.zip` in the same working directory.

**Generated outputs:**
- `output/matching_results.tsv`
- `output/candidate_pairs.tsv`


In [3]:
from pathlib import Path

# ZIP file ka relative path
zip_path = Path("student_resource.zip")

print("File exists:", zip_path.exists())

if zip_path.exists():
    size_gb = zip_path.stat().st_size / (1024**3)
    print(f"File size: {size_gb:.2f} GB")
else:
    print("ZIP file nahi mili. Current folder check karo.")

File exists: True
File size: 1.02 GB


## 1. Load and prepare training data


In [54]:
import zipfile
import pandas as pd

zip_path = "student_resource.zip"

test_source1_path = (
    "student_resource/dataset/test/test_source1.tsv"
)

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(test_source1_path) as file:
        test_source1_chunk = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

display(test_source1_chunk)

,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt,...",France
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...",India
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...",France
5,S1-481669221,Nandlal Kisan LLP,"D-61 Ifs Apartmentmayur Vihar I, New Delhi, Ea...",India
6,S1-909865979,Cure Seafood,"1325 Brooklyn Walk, Issaquah, WA",US
7,S1-280204013,Om Constructions Pvt Ltd,"Karauli, Rajasthan, Karauli, Pani Ki Tanki Ke ...",India
8,S1-742053041,Roongta Sangh,"Bhubaneswar, Sub Plot No.-L6/29, Mahodadhi Bha...",India
9,S1-913506265,Thermal & Fils SASU,"20 Rue Parmentier, Dunkerque, Hauts-de-France",France


In [55]:
print(test_source1_chunk.columns.tolist())
print("Shape:", test_source1_chunk.shape)

['entity_id', 'business_name', 'business_address', 'country']
Shape: (10, 4)


In [56]:
test_source2_path = (
    "student_resource/dataset/test/test_source2.tsv"
)

test_source3_path = (
    "student_resource/dataset/test/test_source3.tsv"
)

with zipfile.ZipFile(zip_path, "r") as z:

    with z.open(test_source2_path) as file:
        test_source2_chunk = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

    with z.open(test_source3_path) as file:
        test_source3_chunk = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

print("Source2 columns:", test_source2_chunk.columns.tolist())
print("Source2 shape:", test_source2_chunk.shape)

print("Source3 columns:", test_source3_chunk.columns.tolist())
print("Source3 shape:", test_source3_chunk.shape)

display(test_source2_chunk)
display(test_source3_chunk)

Source2 columns: ['entity_id', 'business_name', 'business_address', 'country']
Source2 shape: (10, 4)
Source3 columns: ['entity_id', 'business_name', 'business_address', 'country']
Source3 shape: (10, 4)


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karna...",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US
5,S2-910351373,PVT. INDUSTRIES PMB LEASING LIMITED,"SURAT, OFFICE NO. 03038, TRADE HOUSE, NEAR RUS...",India
6,S2-229677112,Clm Agro [Limited],"G-B3/5, T V INDUSTRIAL ESTATE 248/A, S K AHIRE...",India
7,S2-770341988,sci ligue ici parents,"NO. 5 ALLÉE DES HÊTRES, Pornic, Loire-Atlantique",France
8,S2-375151053,A DIAMOND SHORT LLC,"126-B New Line Road, MORRISTOWN, TN",US
9,S2-701289183,Fanni's Dmaigesostcis,"1200 HWY 1187, CROWLEY, TX",US


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V,...",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, ...",India
5,S3-577972146,Perfect Investments Private,"Plot No.99, Flat No.201, Sri Dhama Apts., Road...",India
6,S3-150632472,Pace Mfa PC,"2132 Thomas Run Road, Bel Air, Maryland",US
7,S3-184785497,ஈஸ்டர்ன் கன்சல்டன்சி பிரைவேட் லிமிடெட்,"1410 Sri Mahalakshmi Mandiar 59, Justice Rathi...",India
8,S3-536202929,PU Pvt. Ltd. Services,"TG, Plot No-25, Hyderabad, Qutubullapur, 26, 2...",India
9,S3-560657213,Sun पावर Provision,"143-Alig Flats Rajouri Garden, New Delhi, West...",India


In [57]:
train_source1_path = (
    "student_resource/dataset/train/train_source1.tsv"
)

train_source2_path = (
    "student_resource/dataset/train/train_source2.tsv"
)

train_source3_path = (
    "student_resource/dataset/train/train_source3.tsv"
)

train_ground_truth_path = (
    "student_resource/dataset/train/train_ground_truth.tsv"
)

with zipfile.ZipFile(zip_path, "r") as z:

    with z.open(train_source1_path) as file:
        train_s1_sample = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

    with z.open(train_source2_path) as file:
        train_s2_sample = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

    with z.open(train_source3_path) as file:
        train_s3_sample = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

    with z.open(train_ground_truth_path) as file:
        ground_truth_sample = pd.read_csv(
            file,
            sep="\t",
            nrows=10
        )

print("Train Source1:", train_s1_sample.shape)
print("Train Source2:", train_s2_sample.shape)
print("Train Source3:", train_s3_sample.shape)
print("Ground Truth:", ground_truth_sample.shape)

display(train_s1_sample)
display(train_s2_sample)
display(train_s3_sample)
display(ground_truth_sample)

Train Source1: (10, 4)
Train Source2: (10, 4)
Train Source3: (10, 4)
Ground Truth: (10, 2)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
5,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",India
7,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India
8,S2-721031885,Chavira Platinum Chimera LLC,"282 SAXONY DRIVE, FTT MITCHELL, KY",US
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India
5,S3-578159284,LLC Hernandez Colonial Redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",US
6,S3-121412624,Classic Equity Partners Group,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",US
7,S3-249416830,Ectolumdrex dba X+ Madison Inc,"S03575 Cty Tk M, Town Of Buffalo, WI",US
8,S3-107644605,Animal Hanisch Hospirlg,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",US
9,S3-160217003,Gomez Optimal,"343 Hempstead 161, Hope, Arkansas",US


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [58]:
import zipfile
import pandas as pd

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(train_ground_truth_path) as file:
        ground_truth_df = pd.read_csv(
            file,
            sep="\t"
        )

print("Shape:", ground_truth_df.shape)
print("Columns:", ground_truth_df.columns.tolist())

display(ground_truth_df.head())

Shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [59]:
def parse_matched_ids(value):
    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        entity_id.strip()
        for entity_id in value.split(",")
        if entity_id.strip()
    ]


ground_truth_df["matched_id_list"] = (
    ground_truth_df["matched_entity_ids"]
    .apply(parse_matched_ids)
)

display(
    ground_truth_df[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "matched_id_list"
        ]
    ].head()
)

,source1_entity_id,matched_entity_ids,matched_id_list
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...","[S2-681193310, S2-743505751, S3-775321672, S3-..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...","[S2-249013014, S2-197070651, S3-478195123, S3-..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","[S2-790675320, S2-479876582, S3-878454467]"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","[S2-153058913, S2-24659151, S3-679606215]"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...","[S2-478959098, S2-553508714, S2-625774905, S3-..."


In [60]:
ground_truth_df["match_count"] = (
    ground_truth_df["matched_id_list"].apply(len)
)

print(
    ground_truth_df["match_count"].value_counts().sort_index()
)

match_count
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [61]:
positive_pairs = (
    ground_truth_df[
        ["source1_entity_id", "matched_id_list"]
    ]
    .explode("matched_id_list")
    .rename(
        columns={
            "matched_id_list": "matched_entity_id"
        }
    )
)

positive_pairs = positive_pairs[
    positive_pairs["matched_entity_id"].notna()
].copy()

positive_pairs["label"] = 1

print("Positive pairs:", positive_pairs.shape)

display(positive_pairs.head(10))

Positive pairs: (7638365, 3)


,source1_entity_id,matched_entity_id,label
0,S1-965667,S2-681193310,1
0,S1-965667,S2-743505751,1
0,S1-965667,S3-775321672,1
0,S1-965667,S3-11291185,1
0,S1-965667,S3-860443364,1
1,S1-55344266,S2-249013014,1
1,S1-55344266,S2-197070651,1
1,S1-55344266,S3-478195123,1
1,S1-55344266,S3-384364074,1
2,S1-343815751,S2-790675320,1


In [62]:
positive_pairs["matched_source"] = (
    positive_pairs["matched_entity_id"]
    .str.extract(r"^(S[23])")
)

print(
    positive_pairs["matched_source"]
    .value_counts(dropna=False)
)

matched_source
S3    3944746
S2    3693619
Name: count, dtype: int64


In [63]:
print("Total pairs:", len(positive_pairs))

print(
    "Unique Source1 IDs:",
    positive_pairs["source1_entity_id"].nunique()
)

print(
    "Unique matched IDs:",
    positive_pairs["matched_entity_id"].nunique()
)

print(
    "Duplicate pairs:",
    positive_pairs.duplicated(
        subset=[
            "source1_entity_id",
            "matched_entity_id"
        ]
    ).sum()
)

print("\nLabels:")
print(positive_pairs["label"].value_counts())

print("\nSource distribution:")
print(positive_pairs["matched_source"].value_counts())

Total pairs: 7638365
Unique Source1 IDs: 2083574
Unique matched IDs: 7638365
Duplicate pairs: 0

Labels:
label
1    7638365
Name: count, dtype: int64

Source distribution:
matched_source
S3    3944746
S2    3693619
Name: count, dtype: int64


In [64]:
match_distribution = (
    positive_pairs
    .groupby("source1_entity_id")
    .size()
    .value_counts()
    .sort_index()
)

display(match_distribution)

1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

In [65]:
ground_truth_counts = (
    ground_truth_df["match_count"]
    .value_counts()
    .sort_index()
)

positive_counts = (
    positive_pairs
    .groupby("source1_entity_id")
    .size()
    .value_counts()
    .sort_index()
)

comparison = pd.DataFrame({
    "ground_truth_count": ground_truth_counts,
    "positive_pairs_count": positive_counts
}).fillna(0).astype(int)

display(comparison)

,ground_truth_count,positive_pairs_count
0,123247,0
1,119157,119157
2,375212,375212
3,530841,530841
4,484115,484115
5,321957,321957
6,164868,164868
7,63968,63968
8,18680,18680
9,4205,4205


In [66]:
import zipfile
import pandas as pd

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(train_source1_path) as file:
        train_source1_full = pd.read_csv(
            file,
            sep="\t",
            dtype={
                "entity_id": "string",
                "business_name": "string",
                "business_address": "string",
                "country": "string"
            }
        )

print("Shape:", train_source1_full.shape)
print("Memory usage (MB):",
      train_source1_full.memory_usage(deep=True).sum() / 1024**2)

display(train_source1_full.head())

Shape: (2206821, 4)
Memory usage (MB): 604.4256610870361


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [67]:
positive_pairs_features = positive_pairs.merge(
    train_source1_full[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

positive_pairs_features = (
    positive_pairs_features
    .drop(columns=["entity_id"])
)

print("Shape:", positive_pairs_features.shape)

display(positive_pairs_features.head())

Shape: (7638365, 7)


,source1_entity_id,matched_entity_id,label,matched_source,business_name,business_address,country
0,S1-965667,S2-681193310,1,S2,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US
1,S1-965667,S2-743505751,1,S2,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US
2,S1-965667,S3-775321672,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US
3,S1-965667,S3-11291185,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US
4,S1-965667,S3-860443364,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


In [68]:
s2_ids = set(
    positive_pairs.loc[
        positive_pairs["matched_source"] == "S2",
        "matched_entity_id"
    ]
)

s3_ids = set(
    positive_pairs.loc[
        positive_pairs["matched_source"] == "S3",
        "matched_entity_id"
    ]
)

print("Required Source2 IDs:", len(s2_ids))
print("Required Source3 IDs:", len(s3_ids))

Required Source2 IDs: 3693619
Required Source3 IDs: 3944746


In [69]:
source2_matches = []

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(train_source2_path) as file:

        for chunk in pd.read_csv(
            file,
            sep="\t",
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ],
            dtype="string",
            chunksize=200_000
        ):

            matched_chunk = chunk[
                chunk["entity_id"].isin(s2_ids)
            ]

            if not matched_chunk.empty:
                source2_matches.append(matched_chunk)

train_s2_matched = pd.concat(
    source2_matches,
    ignore_index=True
)

print("Matched Source2 rows:", train_s2_matched.shape)

display(train_s2_matched.head())

Matched Source2 rows: (3693619, 4)


,entity_id,business_name,business_address,country
0,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
1,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
2,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India
3,S2-69394310,heassociates.com,"599 PEASE HILL ROAD, HORICON, NY",US
4,S2-868087865,ASSET BUILDING COALITION LLC,"8219 17RD STREET, LAKE STEVENS, WA",US


In [70]:
source3_matches = []

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(train_source3_path) as file:

        for chunk in pd.read_csv(
            file,
            sep="\t",
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ],
            dtype="string",
            chunksize=200_000
        ):

            matched_chunk = chunk[
                chunk["entity_id"].isin(s3_ids)
            ]

            if not matched_chunk.empty:
                source3_matches.append(matched_chunk)

train_s3_matched = pd.concat(
    source3_matches,
    ignore_index=True
)

print("Matched Source3 rows:", train_s3_matched.shape)

display(train_s3_matched.head())

Matched Source3 rows: (3944746, 4)


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,<NA>,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-578159284,LLC Hernandez Colonial Redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",US


In [71]:
matched_entities = pd.concat(
    [
        train_s2_matched,
        train_s3_matched
    ],
    ignore_index=True
)

matched_entities = matched_entities.rename(
    columns={
        "entity_id": "matched_entity_id",
        "business_name": "matched_business_name",
        "business_address": "matched_business_address",
        "country": "matched_country"
    }
)

print("Shape:", matched_entities.shape)

display(matched_entities.head())

Shape: (7638365, 4)


,matched_entity_id,matched_business_name,matched_business_address,matched_country
0,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
1,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
2,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India
3,S2-69394310,heassociates.com,"599 PEASE HILL ROAD, HORICON, NY",US
4,S2-868087865,ASSET BUILDING COALITION LLC,"8219 17RD STREET, LAKE STEVENS, WA",US


In [72]:
positive_pairs_full = positive_pairs_features.merge(
    matched_entities,
    on="matched_entity_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", positive_pairs_full.shape)

print(
    "Missing matched names:",
    positive_pairs_full["matched_business_name"].isna().sum()
)

print(
    "Missing matched addresses:",
    positive_pairs_full["matched_business_address"].isna().sum()
)

display(positive_pairs_full.head())

Shape: (7638365, 10)
Missing matched names: 15
Missing matched addresses: 337018


,source1_entity_id,matched_entity_id,label,matched_source,business_name,business_address,country,matched_business_name,matched_business_address,matched_country
0,S1-965667,S2-681193310,1,S2,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US,Maure Wilblims Colombier Inc,<NA>,US
1,S1-965667,S2-743505751,1,S2,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US,Maure Williams Colombier,<NA>,US
2,S1-965667,S3-775321672,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
3,S1-965667,S3-11291185,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US
4,S1-965667,S3-860443364,1,S3,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US,Maure Williams Inc Center,<NA>,US


In [73]:
missing_names = positive_pairs_full[
    positive_pairs_full["matched_business_name"].isna()
]

print("Missing name rows:", len(missing_names))

display(
    missing_names[
        [
            "source1_entity_id",
            "matched_entity_id",
            "business_name",
            "business_address",
            "country",
            "matched_business_name",
            "matched_business_address",
            "matched_country"
        ]
    ].head(20)
)

Missing name rows: 15


,source1_entity_id,matched_entity_id,business_name,business_address,country,matched_business_name,matched_business_address,matched_country
10037,S1-729284220,S3-382185873,North Agro Private Limited,"78, Kh No. 138, 1St Floor, Vill: Nawada, Vipin...",India,<NA>,"Kh No. 138, 1St Floor, Vill: Nawada, Vipin Gar...",India
133580,S1-559732496,S3-754148834,Nava Aberdeen,"122 Hwy 173, Lake Arrowhead, CA",US,<NA>,"122 Hwy 173, Lake Arrowhead, CA",US
425699,S1-684348466,S2-599473918,Niva,"7 Lilac Lane, Brookhaven, NY",US,<NA>,"BROOKHAVEN, 7 LILAC LANE, NY",US
572384,S1-514682148,S3-207014979,NX Agri Limited,"C-2 Mansarovar Riico Industrial Area, Jaipur, ...",India,<NA>,"C-2 Mansarovar Riico Industrial Area, Jaipur, ...",India
1333374,S1-679088438,S3-42389827,New Agro Private Limited,"137 - R, Industrial Area B, Near Pahwa Hospita...",India,<NA>,"137 - R, Industrial Area B, Near Pahwa Hospita...",India
1684050,S1-707575573,S3-640593334,North Agro Private Limited,"261/12 F/F Amritpuri B Garhi Jharia Maria, Sou...",India,<NA>,"261/12 F/F Amritpuri B Garhi Jharia Maria, Sou...",India
2136636,S1-15232735,S3-263823274,National Alliance Inc.,"519 Odeneal Street, Dallas, TX",US,<NA>,"519 Odeneal Street, Dallas, TX",US
2744016,S1-757103645,S3-761073618,Nayana Air Private Limited,"H.No. 25/B East Anandpuri, Patna, Bihar",India,<NA>,"H.No. 25/B East Anandpuri, Patna, Bihar",India
3531151,S1-982487128,S3-617170111,Nexance Arcelormittal,"118 Shore Drive, Jarvisburg, NC",US,<NA>,"120 Shore Drive, Jarvisburg, North Carolina",US
3979613,S1-172444211,S2-982925237,Novara Armour L.L.C.,"CT, 11 Ox Yoke Circle, East Hampton",US,<NA>,"11 OX YOKE CIRCLE, EAST HAMPTON, CT",US


In [74]:
print(
    "Duplicate matched IDs:",
    matched_entities["matched_entity_id"].duplicated().sum()
)

Duplicate matched IDs: 0


In [75]:
missing_address_summary = (
    positive_pairs_full.assign(
        address_missing=positive_pairs_full["matched_business_address"].isna()
    )
    .groupby("matched_source")["address_missing"]
    .agg(["sum", "count"])
)

display(missing_address_summary)

,sum,count
matched_source,,
S2,165184,3693619
S3,171834,3944746


In [76]:
missing_pattern = (
    positive_pairs_full.assign(
        name_missing=positive_pairs_full["matched_business_name"].isna(),
        address_missing=positive_pairs_full["matched_business_address"].isna()
    )
    .groupby(["matched_source", "name_missing", "address_missing"])
    .size()
    .reset_index(name="count")
)

display(missing_pattern)

,matched_source,name_missing,address_missing,count
0,S2,False,False,3528433
1,S2,False,True,165184
2,S2,True,False,2
3,S3,False,False,3772899
4,S3,False,True,171834
5,S3,True,False,13


In [77]:
print("Positive pairs:", len(positive_pairs_full))
print("Positive S2 pairs:", (positive_pairs_full["matched_source"] == "S2").sum())
print("Positive S3 pairs:", (positive_pairs_full["matched_source"] == "S3").sum())

Positive pairs: 7638365
Positive S2 pairs: 3693619
Positive S3 pairs: 3944746


In [78]:
# Reproducible sampling
positive_sample = positive_pairs_full.sample(
    n=200_000,
    random_state=42
).copy()

print("Positive sample shape:", positive_sample.shape)
display(positive_sample.head())

Positive sample shape: (200000, 10)


,source1_entity_id,matched_entity_id,label,matched_source,business_name,business_address,country,matched_business_name,matched_business_address,matched_country
5812144,S1-341462203,S2-219553948,1,S2,Baba Technologies Private Limited,"37-90 Squaradeal, Defence Colony Sainikapuri, ...",India,Baba టెక్నాలజీస్ ప్రైవేట్ లిమిటెడ్,"NO 37-90 SQUARADEAL, DEFENCE COLONY SAINIKAPUR...",India
1769771,S1-783955316,S3-656241763,1,S3,Kisan Avas Ltd,"6/161 S 2, Rajender Nagar, Ghaziabad, Uttar Pr...",India,Kisan Avas Ltd.,"उत्तर प्रदेश, 7/161 S 2, Ghaziabad",India
2521555,S1-313221336,S3-872559960,1,S3,Studio 74 Book Store,"2628 Mariposa Drive, Terre Haute, IN",US,Studio 74 BOOK Store,"Indiana, 02628 Mariposa Dr, Terre Ahute",US
3756225,S1-516394603,S2-642842175,1,S2,Bay Creative Stone LLC,"4018 Annapolis Road, Unit APARTMENT B, Haletho...",US,Bay Creative Sthoen LLC,"4018- Annapolis Rd, HALETHORPE, MD",US
315955,S1-887045319,S2-612158384,1,S2,Construction Vr Certifications Pvt Ltd,"H.No1-8-67/P Apiic Kamalanagar, Kushaiguda, Hy...",India,C0nstruction Vr Certhificatidons Pvt Ltd,"#1-8-67/P APIIC KAMALANAGAR, KUSHAIGUDA, RANGA...",India


In [79]:
print("S2 columns:", train_s2_matched.columns.tolist())
print("S3 columns:", train_s3_matched.columns.tolist())

S2 columns: ['entity_id', 'business_name', 'business_address', 'country']
S3 columns: ['entity_id', 'business_name', 'business_address', 'country']


In [80]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Correct column name: entity_id
s2_pool = train_s2_matched["entity_id"].to_numpy()
s3_pool = train_s3_matched["entity_id"].to_numpy()

# Positive pair lookup
positive_pair_set = set(
    zip(
        positive_sample["source1_entity_id"],
        positive_sample["matched_entity_id"]
    )
)

negative_rows = []

for row in positive_sample.itertuples(index=False):
    source1_id = row.source1_entity_id
    source = row.matched_source

    if source == "S2":
        candidate_pool = s2_pool
    else:
        candidate_pool = s3_pool

    # Select a wrong matched ID
    for _ in range(10):
        random_id = rng.choice(candidate_pool)

        if (source1_id, random_id) not in positive_pair_set:
            break

    negative_rows.append({
        "source1_entity_id": source1_id,
        "matched_entity_id": random_id,
        "label": 0,
        "matched_source": source
    })

negative_sample = pd.DataFrame(negative_rows)

print("Negative sample shape:", negative_sample.shape)
display(negative_sample.head())

Negative sample shape: (200000, 4)


,source1_entity_id,matched_entity_id,label,matched_source
0,S1-341462203,S2-866781005,0,S2
1,S1-783955316,S3-658478979,0,S3
2,S1-313221336,S3-835574493,0,S3
3,S1-516394603,S2-262371620,0,S2
4,S1-887045319,S2-330726106,0,S2


In [81]:
# Source1 details merge
negative_features = negative_sample.merge(
    train_source1_full[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
).drop(columns=["entity_id"])

In [82]:
negative_features = negative_features.merge(
    matched_entities,
    on="matched_entity_id",
    how="left",
    validate="many_to_one"
)

print("Negative features shape:", negative_features.shape)

display(negative_features.head())

Negative features shape: (200000, 10)


,source1_entity_id,matched_entity_id,label,matched_source,business_name,business_address,country,matched_business_name,matched_business_address,matched_country
0,S1-341462203,S2-866781005,0,S2,Baba Technologies Private Limited,"37-90 Squaradeal, Defence Colony Sainikapuri, ...",India,साई एंटरप्राइजेज प्राइवेट लिमिटेड,"11 FLOOR-1, MUMBAI II, MUMBAI CITY, Maharashtra",India
1,S1-783955316,S3-658478979,0,S3,Kisan Avas Ltd,"6/161 S 2, Rajender Nagar, Ghaziabad, Uttar Pr...",India,Primary Care Physicians LLC,"1938 Windsor Hill Dr, # G, North Carolina, Mat...",US
2,S1-313221336,S3-835574493,0,S3,Studio 74 Book Store,"2628 Mariposa Drive, Terre Haute, IN",US,Ace Pacific LLC,"6914 371, Walker, Minnesota",US
3,S1-516394603,S2-262371620,0,S2,Bay Creative Stone LLC,"4018 Annapolis Road, Unit APARTMENT B, Haletho...",US,pureindiaproductioncom,"DOOR NO 6-75 H NO -954, SEC-14, SONIPAT, Haryana",India
4,S1-887045319,S2-330726106,0,S2,Construction Vr Certifications Pvt Ltd,"H.No1-8-67/P Apiic Kamalanagar, Kushaiguda, Hy...",India,Revive Nursing Home Pribvate Limited,"Karnataka, NO-601, 6TH FLOOR, B/WH3, PROVIDENT...",India


In [83]:
# Positive samples already contain all required feature columns
positive_training = positive_sample.copy()

# Combine positive and negative pairs
training_pairs = pd.concat(
    [positive_training, negative_features],
    ignore_index=True
)

# Shuffle the data
training_pairs = training_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Final training pairs:", training_pairs.shape)
print("\nLabel distribution:")
print(training_pairs["label"].value_counts())

display(training_pairs.head())

Final training pairs: (400000, 10)

Label distribution:
label
1    200000
0    200000
Name: count, dtype: int64


,source1_entity_id,matched_entity_id,label,matched_source,business_name,business_address,country,matched_business_name,matched_business_address,matched_country
0,S1-69875473,S3-175201808,1,S3,1537 Church Way Apartments LLC,"17 Adams Court, Mansfield, TX",US,1537 Church Way Aartngnts LLC,"17 Adams Court, Mansfield, Texas",US
1,S1-301754052,S2-2295072,1,S2,Surya Marketing Private Limited,"Tamil Nadu, Karaikudi, No-13 Sri Kottai Nachi ...",India,சூர்யா மார்க்கெட்டிங் பிரைவேட் லிமிடெட்,#4-63 NO-13 SRI KOTTAI NACHI COMPLEX KOTTAIYUR...,India
2,S1-805179555,S3-716720244,1,S3,Unique Engineering Private Limited,"C/O Vinay Agarwal, Teli Ke Bazaria Naya Bazar,...",India,Shri Unique Engineering Private Limited,"C/o Vinay Agarwal, Teli Ke Bazaria Naya Bazar,...",India
3,S1-512840882,S2-457828813,1,S2,Jameson Select Electric,"630 Orchard Avenue, Barberton, OH",US,Jameson Select Electicr,"630 ORCHARD AVENUE, BARBERTON, OH",US
4,S1-732706691,S2-734762383,0,S2,Guru Investment Private Limited,"Parnala Gram Panchayat, Village Parnala Tal: L...",India,New Delhi Delhi Limited Services,"HN 703 172, MAITRI APARTMENT PATPARGANJ, NEW D...",India


## 2. Feature engineering and baseline model


In [84]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [85]:
import re
import unicodedata
import numpy as np
from rapidfuzz.fuzz import ratio

In [86]:
def normalize_for_matching(value):
    if pd.isna(value):
        return ""

    value = str(value)
    value = unicodedata.normalize("NFKC", value).casefold()

    cleaned_chars = []

    for char in value:
        category = unicodedata.category(char)

        if category[0] in ("L", "N", "M"):
            cleaned_chars.append(char)
        else:
            cleaned_chars.append(" ")

    value = "".join(cleaned_chars)
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [87]:
training_pairs["source1_name_norm"] = (
    training_pairs["business_name"]
    .map(normalize_for_matching)
)

training_pairs["matched_name_norm"] = (
    training_pairs["matched_business_name"]
    .map(normalize_for_matching)
)

training_pairs["source1_address_norm"] = (
    training_pairs["business_address"]
    .map(normalize_for_matching)
)

training_pairs["matched_address_norm"] = (
    training_pairs["matched_business_address"]
    .map(normalize_for_matching)
)

In [88]:
def calculate_similarity(text1, text2):
    if not text1 or not text2:
        return 0.0

    return ratio(text1, text2) / 100.0

In [89]:
training_pairs["name_similarity"] = [
    calculate_similarity(name1, name2)
    for name1, name2 in zip(
        training_pairs["source1_name_norm"],
        training_pairs["matched_name_norm"]
    )
]

training_pairs["address_similarity"] = [
    calculate_similarity(address1, address2)
    for address1, address2 in zip(
        training_pairs["source1_address_norm"],
        training_pairs["matched_address_norm"]
    )
]

In [90]:
training_pairs["country_match"] = (
    training_pairs["country"].fillna("").str.casefold()
    ==
    training_pairs["matched_country"].fillna("").str.casefold()
).astype(int)

training_pairs["name_missing"] = (
    training_pairs["source1_name_norm"].eq("")
    |
    training_pairs["matched_name_norm"].eq("")
).astype(int)

training_pairs["address_missing"] = (
    training_pairs["source1_address_norm"].eq("")
    |
    training_pairs["matched_address_norm"].eq("")
).astype(int)


In [91]:
feature_columns = [
    "name_similarity",
    "address_similarity",
    "country_match",
    "name_missing",
    "address_missing"
]

display(training_pairs[feature_columns + ["label"]].head(10))

print(training_pairs[feature_columns].describe())

,name_similarity,address_similarity,country_match,name_missing,address_missing,label
0,0.915254,0.947368,1,0,0,1
1,0.085714,0.684211,1,0,0,1
2,0.931507,0.913043,1,0,0,1
3,0.956522,1.000000,1,0,0,1
4,0.412698,0.320000,1,0,0,0
5,0.833333,0.873239,1,0,0,1
6,0.344828,0.315789,0,0,0,0
7,0.666667,0.714932,1,0,0,1
8,0.805556,0.881890,1,0,0,1
9,0.326531,0.376623,1,0,0,0


       name_similarity  address_similarity  country_match   name_missing  \
count    400000.000000       400000.000000  400000.000000  400000.000000   
mean          0.553808            0.532030       0.760238       0.000003   
std           0.306126            0.284299       0.426939       0.001581   
min           0.000000            0.000000       0.000000       0.000000   
25%           0.301887            0.309091       1.000000       0.000000   
50%           0.428571            0.406780       1.000000       0.000000   
75%           0.875000            0.827586       1.000000       0.000000   
max           1.000000            1.000000       1.000000       1.000000   

       address_missing  
count    400000.000000  
mean          0.043708  
std           0.204444  
min           0.000000  
25%           0.000000  
50%           0.000000  
75%           0.000000  
max           1.000000  


In [92]:
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "name_similarity",
    "address_similarity",
    "country_match",
    "name_missing",
    "address_missing"
]

X = training_pairs[feature_columns].astype("float32")
y = training_pairs["label"].astype("int8")

groups = training_pairs["source1_entity_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Training labels:")
print(y_train.value_counts())
print("\nValidation labels:")
print(y_val.value_counts())

Training shape: (320048, 5)
Validation shape: (79952, 5)
Training labels:
label
1    160024
0    160024
Name: count, dtype: int64

Validation labels:
label
1    39976
0    39976
Name: count, dtype: int64


In [93]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed!")

F:\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "F:\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "F:\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "F:\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "F:\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^

Model training completed!


In [94]:
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

val_probabilities = model.predict_proba(X_val)[:, 1]

threshold = 0.5

val_predictions = (
    val_probabilities >= threshold
).astype(int)

print(classification_report(
    y_val,
    val_predictions,
    digits=4
))

print("Precision:", precision_score(y_val, val_predictions))
print("Recall:", recall_score(y_val, val_predictions))
print("F1:", f1_score(y_val, val_predictions))

              precision    recall  f1-score   support

           0     0.9832    0.9917    0.9874     39976
           1     0.9916    0.9831    0.9873     39976

    accuracy                         0.9874     79952
   macro avg     0.9874    0.9874    0.9874     79952
weighted avg     0.9874    0.9874    0.9874     79952

Precision: 0.9916229309648769
Recall: 0.9830898539123474
F1: 0.9873379559843232


In [95]:
from sklearn.metrics import fbeta_score

f05 = fbeta_score(
    y_val,
    val_predictions,
    beta=0.5
)

print("F0.5 Score:", f05)

F0.5 Score: 0.9899044855519284


In [96]:
from sklearn.metrics import precision_score, recall_score, fbeta_score
import pandas as pd

threshold_results = []

for threshold in np.arange(0.50, 1.00, 0.05):
    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f0.5": f05
    })

threshold_results_df = pd.DataFrame(threshold_results)

display(
    threshold_results_df.sort_values(
        "f0.5",
        ascending=False
    )
)


,threshold,precision,recall,f0.5
7,0.85,0.997637,0.971458,0.992289
6,0.80,0.996625,0.975160,0.992257
8,0.90,0.998425,0.967030,0.991984
5,0.75,0.995517,0.977687,0.991899
4,0.70,0.995017,0.978962,0.991764
3,0.65,0.994440,0.979863,0.991490
2,0.60,0.993315,0.981339,0.990897
9,0.95,0.999244,0.958675,0.990858
1,0.55,0.992518,0.982189,0.990435
0,0.50,0.991623,0.983090,0.989904


In [97]:
best_threshold = 0.85

print("Provisional threshold:", best_threshold)





Provisional threshold: 0.85


## 3. Hard negatives and improved model


In [140]:
positive_pair_set = set(
    zip(
        positive_pairs["source1_entity_id"],
        positive_pairs["matched_entity_id"]
    )
)

print("Positive pair count:", len(positive_pair_set))

Positive pair count: 7638365


In [141]:
hard_source1 = train_source1_full.sample(
    n=100_000,
    random_state=42
).copy()

hard_source1 = hard_source1.rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "source1_business_name",
    "business_address": "source1_business_address",
    "country": "source1_country"
})

hard_source1["source1_name_norm"] = (
    hard_source1["source1_business_name"]
    .map(normalize_for_matching)
)

hard_source1["source1_country_norm"] = (
    hard_source1["source1_country"]
    .fillna("")
    .astype(str)
    .str.casefold()
)

hard_source1["name_key"] = (
    hard_source1["source1_country_norm"]
    + "||"
    + hard_source1["source1_name_norm"]
)

print("Sample Source1 rows:", len(hard_source1))

Sample Source1 rows: 100000


In [142]:
hard_matched = matched_entities.copy()

hard_matched["matched_name_norm"] = (
    hard_matched["matched_business_name"]
    .map(normalize_for_matching)
)

hard_matched["matched_country_norm"] = (
    hard_matched["matched_country"]
    .fillna("")
    .astype(str)
    .str.casefold()
)

hard_matched["name_key"] = (
    hard_matched["matched_country_norm"]
    + "||"
    + hard_matched["matched_name_norm"]
)

hard_matched = hard_matched[
    hard_matched["name_key"].ne("||")
].copy()

print("Matched entities:", len(hard_matched))

Matched entities: 7638365


In [143]:
hard_candidates = hard_source1.merge(
    hard_matched[
        [
            "matched_entity_id",
            "matched_business_name",
            "matched_business_address",
            "matched_country",
            "name_key"
        ]
    ],
    on="name_key",
    how="inner"
)

print("Initial hard candidates:", len(hard_candidates))

Initial hard candidates: 920932


In [144]:
hard_candidates["pair"] = list(
    zip(
        hard_candidates["source1_entity_id"],
        hard_candidates["matched_entity_id"]
    )
)

hard_candidates = hard_candidates[
    ~hard_candidates["pair"].isin(positive_pair_set)
].copy()

hard_candidates = hard_candidates.drop(columns=["pair"])

print("Hard negative candidates:", len(hard_candidates))

Hard negative candidates: 845062


In [145]:
hard_candidates["source1_address_norm"] = (
    hard_candidates["source1_business_address"]
    .map(normalize_for_matching)
)

hard_candidates["matched_address_norm"] = (
    hard_candidates["matched_business_address"]
    .map(normalize_for_matching)
)

hard_candidates["name_similarity"] = 1.0

hard_candidates["address_similarity"] = [
    calculate_similarity(address1, address2)
    for address1, address2 in zip(
        hard_candidates["source1_address_norm"],
        hard_candidates["matched_address_norm"]
    )
]

hard_candidates["country_match"] = 1

In [146]:
hard_negative_candidates = hard_candidates[
    hard_candidates["address_similarity"] < 0.50
].copy()

print(
    "Strong hard negative candidates:",
    len(hard_negative_candidates)
)

Strong hard negative candidates: 832852


In [147]:
display(
    hard_negative_candidates[
        [
            "source1_entity_id",
            "matched_entity_id",
            "source1_business_name",
            "matched_business_name",
            "source1_business_address",
            "matched_business_address",
            "address_similarity"
        ]
    ].head(20)
)

,source1_entity_id,matched_entity_id,source1_business_name,matched_business_name,source1_business_address,matched_business_address,address_similarity
0,S1-53356671,S2-224665641,Pediatric Medicine PLLC,Pediatric Medicine PLLC,"4850 20, Otisco, NY","616 Walnut Creek Rd, DANVILLE, VA",0.208333
1,S1-53356671,S2-444730103,Pediatric Medicine PLLC,pediatric medicine pllc,"4850 20, Otisco, NY","3025 LEONARD ROAD, LEXINGTON, NC",0.340426
3,S1-53356671,S3-104607810,Pediatric Medicine PLLC,Pediatric Medicine (PLLC),"4850 20, Otisco, NY","147-55 28 Avenue, Flushing, New York",0.352941
4,S1-53356671,S3-696248950,Pediatric Medicine PLLC,Pediatric Medicine Pllc,"4850 20, Otisco, NY","2314. Molly Lane, Dunlap, Illinois",0.250000
5,S1-938947364,S2-937244620,General Design Innovations LLC,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ","CHICAGO HEIGHTS, IL, 22332. MERRILL AVENUE",0.312500
6,S1-938947364,S3-984985270,General Design Innovations LLC,general design innovations llc,"7241 Osage Avenue, Mesa, AZ","Merrill Ave, Chicago Heights, Illinois",0.295082
7,S1-938947364,S3-874480734,General Design Innovations LLC,-- General Design Innovations Llc,"7241 Osage Avenue, Mesa, AZ","22332 Merrill Ave, Chiago Heights, Illinois",0.333333
11,S1-758070915,S2-253264439,National Investments LLC,NATIONAL INVESTMENTS (LLC),"1641 Virginia Lane, Hueytown, AL","FELTS STATION ROAD, MEMPHIS, TN",0.305085
12,S1-758070915,S3-169185870,National Investments LLC,National Investments Llc,"1641 Virginia Lane, Hueytown, AL","01481 8, Town Of Armstrong Creek, Wisconsin",0.338028
13,S1-758070915,S3-234789072,National Investments LLC,National Investments LLC,"1641 Virginia Lane, Hueytown, AL","3899 Felts Station Road, Memphis, Tennessee",0.253521


In [148]:
# Source1 normalized columns
hard_candidates["source1_name_norm"] = (
    hard_candidates["source1_business_name"]
    .map(normalize_for_matching)
)

hard_candidates["source1_address_norm"] = (
    hard_candidates["source1_business_address"]
    .map(normalize_for_matching)
)

# Matched entity normalized columns
hard_candidates["matched_name_norm"] = (
    hard_candidates["matched_business_name"]
    .map(normalize_for_matching)
)

hard_candidates["matched_address_norm"] = (
    hard_candidates["matched_business_address"]
    .map(normalize_for_matching)
)

In [149]:
# ==========================================
# STEP 8: Prepare clean hard negatives
# ==========================================

# 1. Create normalized columns
hard_candidates["source1_name_norm"] = (
    hard_candidates["source1_business_name"]
    .map(normalize_for_matching)
)

hard_candidates["source1_address_norm"] = (
    hard_candidates["source1_business_address"]
    .map(normalize_for_matching)
)

hard_candidates["matched_name_norm"] = (
    hard_candidates["matched_business_name"]
    .map(normalize_for_matching)
)

hard_candidates["matched_address_norm"] = (
    hard_candidates["matched_business_address"]
    .map(normalize_for_matching)
)


# 2. Calculate name similarity
hard_candidates["name_similarity"] = [
    calculate_similarity(name1, name2)
    for name1, name2 in zip(
        hard_candidates["source1_name_norm"],
        hard_candidates["matched_name_norm"]
    )
]


# 3. Calculate address similarity
hard_candidates["address_similarity"] = [
    calculate_similarity(address1, address2)
    for address1, address2 in zip(
        hard_candidates["source1_address_norm"],
        hard_candidates["matched_address_norm"]
    )
]


# 4. Missing-value flags
hard_candidates["name_missing"] = (
    hard_candidates["source1_name_norm"].eq("")
    |
    hard_candidates["matched_name_norm"].eq("")
).astype("int8")

hard_candidates["address_missing"] = (
    hard_candidates["source1_address_norm"].eq("")
    |
    hard_candidates["matched_address_norm"].eq("")
).astype("int8")


# 5. Country match
hard_candidates["country_match"] = (
    hard_candidates["source1_country"].fillna("").astype(str).str.casefold()
    ==
    hard_candidates["matched_country"].fillna("").astype(str).str.casefold()
).astype("int8")


# 6. Select clean hard negatives
clean_hard_negatives = hard_candidates[
    (hard_candidates["address_missing"] == 0)
    &
    (hard_candidates["address_similarity"] < 0.50)
].copy()


print("Total hard candidates:", len(hard_candidates))
print("Clean hard negatives:", len(clean_hard_negatives))
print()
print("Address missing distribution:")
print(hard_candidates["address_missing"].value_counts())

Total hard candidates: 845062
Clean hard negatives: 812939

Address missing distribution:
address_missing
0    825149
1     19913
Name: count, dtype: int64


In [150]:
display(
    clean_hard_negatives[
        [
            "source1_entity_id",
            "matched_entity_id",
            "source1_business_name",
            "matched_business_name",
            "source1_business_address",
            "matched_business_address",
            "name_similarity",
            "address_similarity",
            "country_match"
        ]
    ].head(20)
)

,source1_entity_id,matched_entity_id,source1_business_name,matched_business_name,source1_business_address,matched_business_address,name_similarity,address_similarity,country_match
0,S1-53356671,S2-224665641,Pediatric Medicine PLLC,Pediatric Medicine PLLC,"4850 20, Otisco, NY","616 Walnut Creek Rd, DANVILLE, VA",1.0,0.208333,1
1,S1-53356671,S2-444730103,Pediatric Medicine PLLC,pediatric medicine pllc,"4850 20, Otisco, NY","3025 LEONARD ROAD, LEXINGTON, NC",1.0,0.340426,1
3,S1-53356671,S3-104607810,Pediatric Medicine PLLC,Pediatric Medicine (PLLC),"4850 20, Otisco, NY","147-55 28 Avenue, Flushing, New York",1.0,0.352941,1
4,S1-53356671,S3-696248950,Pediatric Medicine PLLC,Pediatric Medicine Pllc,"4850 20, Otisco, NY","2314. Molly Lane, Dunlap, Illinois",1.0,0.250000,1
5,S1-938947364,S2-937244620,General Design Innovations LLC,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ","CHICAGO HEIGHTS, IL, 22332. MERRILL AVENUE",1.0,0.312500,1
6,S1-938947364,S3-984985270,General Design Innovations LLC,general design innovations llc,"7241 Osage Avenue, Mesa, AZ","Merrill Ave, Chicago Heights, Illinois",1.0,0.295082,1
7,S1-938947364,S3-874480734,General Design Innovations LLC,-- General Design Innovations Llc,"7241 Osage Avenue, Mesa, AZ","22332 Merrill Ave, Chiago Heights, Illinois",1.0,0.333333,1
11,S1-758070915,S2-253264439,National Investments LLC,NATIONAL INVESTMENTS (LLC),"1641 Virginia Lane, Hueytown, AL","FELTS STATION ROAD, MEMPHIS, TN",1.0,0.305085,1
12,S1-758070915,S3-169185870,National Investments LLC,National Investments Llc,"1641 Virginia Lane, Hueytown, AL","01481 8, Town Of Armstrong Creek, Wisconsin",1.0,0.338028,1
13,S1-758070915,S3-234789072,National Investments LLC,National Investments LLC,"1641 Virginia Lane, Hueytown, AL","3899 Felts Station Road, Memphis, Tennessee",1.0,0.253521,1


In [151]:
# Select 100,000 hard negatives
hard_negative_sample = clean_hard_negatives.sample(
    n=min(100_000, len(clean_hard_negatives)),
    random_state=42
).copy()

hard_negative_sample["label"] = 0

print("Selected hard negatives:", len(hard_negative_sample))

Selected hard negatives: 100000


In [152]:
hard_negative_training = hard_negative_sample[
    [
        "source1_entity_id",
        "matched_entity_id",
        "label",
        "source1_name_norm",
        "matched_name_norm",
        "source1_address_norm",
        "matched_address_norm",
        "name_similarity",
        "address_similarity",
        "country_match",
        "name_missing",
        "address_missing"
    ]
].copy()

print("Hard-negative training shape:", hard_negative_training.shape)

Hard-negative training shape: (100000, 12)


In [153]:
base_training = training_pairs[
    [
        "source1_entity_id",
        "matched_entity_id",
        "label",
        "source1_name_norm",
        "matched_name_norm",
        "source1_address_norm",
        "matched_address_norm",
        "name_similarity",
        "address_similarity",
        "country_match",
        "name_missing",
        "address_missing"
    ]
].copy()

improved_training = pd.concat(
    [
        base_training,
        hard_negative_training
    ],
    ignore_index=True
)

improved_training = improved_training.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Improved training shape:", improved_training.shape)
print()
print("Label distribution:")
print(improved_training["label"].value_counts())

Improved training shape: (500000, 12)

Label distribution:
label
0    300000
1    200000
Name: count, dtype: int64


In [154]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Features
feature_columns = [
    "name_similarity",
    "address_similarity",
    "country_match",
    "name_missing",
    "address_missing"
]

X_improved = improved_training[feature_columns].astype("float32")
y_improved = improved_training["label"].astype("int8")

# Group by Source1 entity
groups_improved = improved_training["source1_entity_id"]

# Train-validation split
splitter_improved = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter_improved.split(
        X_improved,
        y_improved,
        groups=groups_improved
    )
)

X_train_improved = X_improved.iloc[train_idx]
X_val_improved = X_improved.iloc[val_idx]

y_train_improved = y_improved.iloc[train_idx]
y_val_improved = y_improved.iloc[val_idx]

print("Training shape:", X_train_improved.shape)
print("Validation shape:", X_val_improved.shape)

Training shape: (400307, 5)
Validation shape: (99693, 5)


In [155]:
improved_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

improved_model.fit(
    X_train_improved,
    y_train_improved
)

print("Improved model training completed!")

Improved model training completed!


In [156]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict probabilities
val_probabilities = improved_model.predict_proba(
    X_val_improved
)[:, 1]

# Default threshold
val_predictions = (
    val_probabilities >= 0.5
).astype("int8")

print("Classification Report:")
print(
    classification_report(
        y_val_improved,
        val_predictions,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_improved,
        val_predictions
    )
)

Classification Report:
              precision    recall  f1-score   support

           0     0.9793    0.9944    0.9868     59727
           1     0.9914    0.9686    0.9799     39966

    accuracy                         0.9841     99693
   macro avg     0.9854    0.9815    0.9834     99693
weighted avg     0.9842    0.9841    0.9840     99693


Confusion Matrix:
[[59392   335]
 [ 1253 38713]]


In [157]:
from sklearn.metrics import fbeta_score
import numpy as np

threshold_results = []

for threshold in np.arange(0.50, 1.00, 0.01):
    predictions = (
        val_probabilities >= threshold
    ).astype("int8")

    score = fbeta_score(
        y_val_improved,
        predictions,
        beta=0.5
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "f0.5": score
    })

threshold_results_df = pd.DataFrame(threshold_results)

best_result = threshold_results_df.loc[
    threshold_results_df["f0.5"].idxmax()
]

print("Best threshold:", best_result["threshold"])
print("Best pair-level F0.5:", best_result["f0.5"])

Best threshold: 0.75
Best pair-level F0.5: 0.9886549900518541


## 4. Final test inference and output generation


In [163]:
# ================= FINAL END-TO-END INFERENCE =================
# Run this ONE cell after the training cells. It creates both required TSV files.

import os, csv, math, gc
from collections import defaultdict
import numpy as np
import pandas as pd

assert "improved_model" in globals(), "improved_model missing: run the training cells first."
assert "zip_path" in globals(), "zip_path missing."

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use the threshold selected during validation; keep a conservative floor for precision.
FINAL_THRESHOLD = max(0.75, float(best_result["threshold"]) if "best_result" in globals() else 0.75)
print("Final threshold:", FINAL_THRESHOLD)

# Load complete test Source1.
with zipfile.ZipFile(zip_path, "r") as z:
    test_source1 = pd.read_csv(
        z.open("student_resource/dataset/test/test_source1.tsv"),
        sep="\t", dtype="string", keep_default_na=False
    )

for df in [test_source1]:
    df["name_norm"] = df["business_name"].map(normalize_for_matching)
    df["address_norm"] = df["business_address"].map(normalize_for_matching)
    df["country_norm"] = df["country"].fillna("").astype(str).str.casefold().str.strip()
    df["name_key"] = df["country_norm"] + "||" + df["name_norm"]
    df["address_key"] = df["country_norm"] + "||" + df["address_norm"]

# Build compact blocking indexes from COMPLETE Source2 + Source3.
# Only non-empty exact normalized name/address keys are indexed.
name_index = defaultdict(list)
address_index = defaultdict(list)
record_store = {}

with zipfile.ZipFile(zip_path, "r") as z:
    for source_file in [
        "student_resource/dataset/test/test_source2.tsv",
        "student_resource/dataset/test/test_source3.tsv"
    ]:
        print("Indexing:", source_file)
        with z.open(source_file) as f:
            for chunk in pd.read_csv(
                f, sep="\t", dtype="string", keep_default_na=False,
                chunksize=100_000
            ):
                chunk["name_norm"] = chunk["business_name"].map(normalize_for_matching)
                chunk["address_norm"] = chunk["business_address"].map(normalize_for_matching)
                chunk["country_norm"] = chunk["country"].fillna("").astype(str).str.casefold().str.strip()
                chunk["name_key"] = chunk["country_norm"] + "||" + chunk["name_norm"]
                chunk["address_key"] = chunk["country_norm"] + "||" + chunk["address_norm"]

                for r in chunk.itertuples(index=False):
                    rid = str(r.entity_id)
                    record_store[rid] = (
                        str(r.business_name), str(r.business_address), str(r.country),
                        str(r.name_norm), str(r.address_norm), str(r.country_norm)
                    )
                    if r.name_norm:
                        name_index[r.name_key].append(rid)
                    if r.address_norm:
                        address_index[r.address_key].append(rid)

print("Indexed records:", len(record_store))
print("Name keys:", len(name_index), "Address keys:", len(address_index))

# Candidate generation + model inference in batches.
# Exact name/address blocking is intentionally precision-oriented.
match_rows = []
candidate_rows = []
feature_columns = [
    "name_similarity", "address_similarity", "country_match",
    "name_missing", "address_missing"
]

for start in range(0, len(test_source1), 10_000):
    s1batch = test_source1.iloc[start:start+10_000]
    batch_candidates = []
    batch_candidate_map = {}

    for r in s1batch.itertuples(index=False):
        sid = str(r.entity_id)
        ids = set()
        if r.name_norm:
            ids.update(name_index.get(r.name_key, []))
        if r.address_norm:
            ids.update(address_index.get(r.address_key, []))
        ids = sorted(ids)
        batch_candidate_map[sid] = ids
        for cid in ids:
            rec = record_store[cid]
            batch_candidates.append({
                "source1_entity_id": sid,
                "matched_entity_id": cid,
                "source1_name_norm": r.name_norm,
                "matched_name_norm": rec[3],
                "source1_address_norm": r.address_norm,
                "matched_address_norm": rec[4],
                "source1_country_norm": r.country_norm,
                "matched_country_norm": rec[5]
            })

    # Save exact final candidates fed into the model.
    for sid, ids in batch_candidate_map.items():
        candidate_rows.append({
            "source1_entity_id": sid,
            "candidate_entity_ids": ",".join(ids)
        })

    if batch_candidates:
        pairs = pd.DataFrame(batch_candidates)
        pairs["name_similarity"] = [
            calculate_similarity(a, b)
            for a, b in zip(pairs["source1_name_norm"], pairs["matched_name_norm"])
        ]
        pairs["address_similarity"] = [
            calculate_similarity(a, b)
            for a, b in zip(pairs["source1_address_norm"], pairs["matched_address_norm"])
        ]
        pairs["country_match"] = (
            pairs["source1_country_norm"] == pairs["matched_country_norm"]
        ).astype("int8")
        pairs["name_missing"] = pairs["source1_name_norm"].eq("").astype("int8")
        pairs["address_missing"] = pairs["source1_address_norm"].eq("").astype("int8")

        probs = improved_model.predict_proba(pairs[feature_columns].astype("float32"))[:, 1]
        pairs["probability"] = probs

        # Require both exact blocking and a conservative model probability.
        selected = pairs[pairs["probability"] >= FINAL_THRESHOLD]
        selected_by_s1 = defaultdict(list)
        for rr in selected.itertuples(index=False):
            selected_by_s1[str(rr.source1_entity_id)].append(str(rr.matched_entity_id))
    else:
        selected_by_s1 = defaultdict(list)

    for r in s1batch.itertuples(index=False):
        sid = str(r.entity_id)
        ids = sorted(set(selected_by_s1.get(sid, [])))
        match_rows.append({
            "source1_entity_id": sid,
            "matched_entity_ids": ",".join(ids)
        })

    if start % 100_000 == 0:
        print(f"Processed Source1 rows: {min(start+10_000, len(test_source1)):,}/{len(test_source1):,}")

# Guarantee exactly one output row per test Source1 ID and preserve test order.
matching_df = pd.DataFrame(match_rows).drop_duplicates("source1_entity_id", keep="first")
candidate_df = pd.DataFrame(candidate_rows).drop_duplicates("source1_entity_id", keep="first")

matching_df = test_source1[["entity_id"]].rename(columns={"entity_id":"source1_entity_id"}).merge(
    matching_df, on="source1_entity_id", how="left"
)
candidate_df = test_source1[["entity_id"]].rename(columns={"entity_id":"source1_entity_id"}).merge(
    candidate_df, on="source1_entity_id", how="left"
)
matching_df["matched_entity_ids"] = matching_df["matched_entity_ids"].fillna("")
candidate_df["candidate_entity_ids"] = candidate_df["candidate_entity_ids"].fillna("")

matching_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
matching_df.to_csv(matching_path, sep="\t", index=False, quoting=csv.QUOTE_MINIMAL)
candidate_df.to_csv(candidate_path, sep="\t", index=False, quoting=csv.QUOTE_MINIMAL)

print("\nDONE")
print(matching_path, matching_df.shape)
print(candidate_path, candidate_df.shape)
print("Non-empty predictions:", (matching_df["matched_entity_ids"] != "").sum())
print("Non-empty candidate rows:", (candidate_df["candidate_entity_ids"] != "").sum())

# Local official validation.
import subprocess, sys
validator = "student_resource/utils/validate_submission.py"
if os.path.exists(validator):
    result = subprocess.run([
        sys.executable, validator,
        "--matching", matching_path,
        "--candidate", candidate_path,
        "--test-dir", "student_resource/dataset/test"
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Validator not found at:", validator)


Final threshold: 0.75
Indexing: student_resource/dataset/test/test_source2.tsv
Indexing: student_resource/dataset/test/test_source3.tsv
Indexed records: 9969589
Name keys: 7658877 Address keys: 8485550
Processed Source1 rows: 10,000/1,732,544
Processed Source1 rows: 110,000/1,732,544
Processed Source1 rows: 210,000/1,732,544
Processed Source1 rows: 310,000/1,732,544
Processed Source1 rows: 410,000/1,732,544
Processed Source1 rows: 510,000/1,732,544
Processed Source1 rows: 610,000/1,732,544
Processed Source1 rows: 710,000/1,732,544
Processed Source1 rows: 810,000/1,732,544
Processed Source1 rows: 910,000/1,732,544
Processed Source1 rows: 1,010,000/1,732,544
Processed Source1 rows: 1,110,000/1,732,544
Processed Source1 rows: 1,210,000/1,732,544
Processed Source1 rows: 1,310,000/1,732,544
Processed Source1 rows: 1,410,000/1,732,544
Processed Source1 rows: 1,510,000/1,732,544
Processed Source1 rows: 1,610,000/1,732,544
Processed Source1 rows: 1,710,000/1,732,544

DONE
output\matching_resul

## 5. Output validation checks


In [167]:
import pandas as pd

matching = pd.read_csv(
    "output/matching_results.tsv",
    sep="\t",
    dtype=str
)

candidate = pd.read_csv(
    "output/candidate_pairs.tsv",
    sep="\t",
    dtype=str
)

print("Matching shape:", matching.shape)
print("Candidate shape:", candidate.shape)

print("\nMatching columns:")
print(matching.columns.tolist())

print("\nCandidate columns:")
print(candidate.columns.tolist())

print("\nDuplicate Source1 IDs:")
print("Matching:", matching["source1_entity_id"].duplicated().sum())
print("Candidate:", candidate["source1_entity_id"].duplicated().sum())

print("\nMissing values:")
print("Matching:\n", matching.isna().sum())
print("Candidate:\n", candidate.isna().sum())

Matching shape: (1732544, 2)
Candidate shape: (1732544, 2)

Matching columns:
['source1_entity_id', 'matched_entity_ids']

Candidate columns:
['source1_entity_id', 'candidate_entity_ids']

Duplicate Source1 IDs:
Matching: 0
Candidate: 0

Missing values:
Matching:
 source1_entity_id          0
matched_entity_ids    598293
dtype: int64
Candidate:
 source1_entity_id            0
candidate_entity_ids    393607
dtype: int64


In [169]:
# Check that every matching prediction is included in its candidate set

def split_ids(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    }

candidate_map = dict(
    zip(
        candidate["source1_entity_id"],
        candidate["candidate_entity_ids"].fillna("")
    )
)

violations = 0

for _, row in matching.iterrows():
    source1_id = row["source1_entity_id"]

    matched_ids = split_ids(row["matched_entity_ids"])
    candidate_ids = split_ids(candidate_map.get(source1_id, ""))

    if not matched_ids.issubset(candidate_ids):
        violations += 1

print("Matching not in candidates:", violations)

Matching not in candidates: 0


In [170]:
# Check duplicate IDs inside predictions and candidates

def has_duplicate_ids(value):
    if pd.isna(value) or str(value).strip() == "":
        return False

    ids = [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

    return len(ids) != len(set(ids))


matching_duplicates = matching["matched_entity_ids"].apply(
    has_duplicate_ids
).sum()

candidate_duplicates = candidate["candidate_entity_ids"].apply(
    has_duplicate_ids
).sum()

print("Rows with duplicate matched IDs:", matching_duplicates)
print("Rows with duplicate candidate IDs:", candidate_duplicates)

Rows with duplicate matched IDs: 0
Rows with duplicate candidate IDs: 0
